# Preprocessing & Feature Engineering

Translates the signals surfaced in `eda-presentation.ipynb` into a reproducible,
**leakage-safe** preprocessing + feature-engineering pipeline that emits
modeling-ready datasets.

**EDA signals encoded here**
- **OverTime** is the single strongest driver (30% vs 10%).
- Heavy **business travel** (~25% vs 8%), **single** marital status (~25% vs ~12%),
  and **Sales Rep / Lab Technician** roles run hot.
- **Low engagement / work-life balance** roughly doubles risk.
- **Lower pay**, especially *below peers in the same role/level*, drives exits.
- **Stalled promotions** (0 yrs *and* 6+ yrs) carry higher risk.
- Class imbalance ~16% &rarr; judged by Recall (Yes), not accuracy.

**Leakage guardrails:** the stratified split happens *before* any statistic is
computed. Every median / percentile / scaler is fit on **train only** and applied
to test by mapping the stored train value.

## 1. Setup & load

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

dataset = 'WA_Fn-UseC_-HR-Employee-Attrition.csv'

filepath = '../' + dataset

df = pd.read_csv(filepath)

print('Shape:', df.shape)
print('Missing values:', int(df.isna().sum().sum()))
print('Duplicate rows:', int(df.duplicated().sum()))
df.head(3)

## 2. Cleaning

Drop the four non-informative columns flagged in EDA (`EmployeeCount`,
`EmployeeNumber`, `Over18`, `StandardHours` are constant or pure IDs), then
split off the target.

In [ ]:
DROP_COLS = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
df = df.drop(columns=DROP_COLS)

y = (df['Attrition'] == 'Yes').astype(int)
X = df.drop(columns=['Attrition'])

print('Dropped:', DROP_COLS)
print('Target balance:')
print(y.value_counts(normalize=True).rename({0: 'Stayed (0)', 1: 'Left (1)'}).round(3))
print('Feature frame:', X.shape)

## 3. Categorical encoding (pre-split, deterministic only)

Only mappings that are **independent of data statistics** run before the split:
- **Binary**: `Gender` (Female=0, Male=1), `OverTime` (No=0, Yes=1).
- **Ordinal**: `BusinessTravel` (Non-Travel=0, Travel_Rarely=1, Travel_Frequently=2).
- **One-hot** (`drop_first` reference category): `Department`, `EducationField`,
  `JobRole`, `MaritalStatus`.

Raw `JobRole` / `MaritalStatus` are stashed as `_JobRole` / `_MaritalStatus`
helper columns first — the engineered features in step 5 need them. They are
dropped again at the end of step 5.

In [ ]:
# Binary
X['Gender'] = X['Gender'].map({'Female': 0, 'Male': 1}).astype(int)
X['OverTime'] = X['OverTime'].map({'No': 0, 'Yes': 1}).astype(int)

# Ordinal
X['BusinessTravel'] = X['BusinessTravel'].map(
    {'Non-Travel': 0, 'Travel_Rarely': 1, 'Travel_Frequently': 2}
).astype(int)

# Stash raw helpers needed for engineered features (peer-income groups, Single flag)
X['_JobRole'] = X['JobRole'].astype('object')
X['_MaritalStatus'] = X['MaritalStatus'].astype('object')

# One-hot with dropped reference category
X = pd.get_dummies(
    X,
    columns=['Department', 'EducationField', 'JobRole', 'MaritalStatus'],
    drop_first=True,
    dtype=int,
)

print('Encoded feature frame:', X.shape)
print('Sanity — MaritalStatus_Single present:', 'MaritalStatus_Single' in X.columns)

## 4. Stratified train/test split

`test_size=0.20`, `stratify=y` preserves the ~16% attrition rate in both splits.
**Everything downstream is fit on `X_train` only.**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]} rows | attrition {y_train.mean():.3f}')
print(f'Test:  {X_test.shape[0]} rows | attrition {y_test.mean():.3f}')

## 5. Feature engineering (EDA-driven, train-fit stats)

Every threshold / peer median is computed on **`X_train`** and applied to both
splits. Ratio features guard against divide-by-zero (`TotalWorkingYears` is 0 for
11 employees).

| Group | Features |
|---|---|
| OverTime interactions | `Single_OT`, `FreqTravel_OT`, `IsHighRisk` |
| Pay vs peers | `PeerRelativeIncome`, `IsLowIncome`, `IncomePerLevel` |
| Tenure / career | `EarlyTenure`, `TenureRatio`, `PromotionOverdue`, `JobHoppingIndex` |
| Engagement / well-being | `SatisfactionComposite`, `EngagementScore`, `IsLowEngagement` |

In [ ]:
# --- Train-derived statistics (fit on TRAIN only) ---
low_income_thresh = X_train['MonthlyIncome'].quantile(0.33)
peer_median = X_train.groupby(['_JobRole', 'JobLevel'])['MonthlyIncome'].median()
global_income_median = X_train['MonthlyIncome'].median()
print(f'Low-income threshold (train 33rd pct): {low_income_thresh:,.0f}')
print(f'Peer groups (JobRole x JobLevel): {peer_median.shape[0]}')


def engineer(X):
    """Apply EDA-driven features using train-fit stats. Returns a new frame."""
    X = X.copy()

    # OverTime interactions (strongest signal)
    X['Single_OT'] = (X['MaritalStatus_Single'] * X['OverTime']).astype(int)
    X['FreqTravel_OT'] = ((X['BusinessTravel'] == 2).astype(int) * X['OverTime']).astype(int)

    # Pay vs peers (below-peer pay = flight risk)
    peer = X.set_index(['_JobRole', 'JobLevel']).index.map(peer_median)
    peer = pd.Series(peer, index=X.index).fillna(global_income_median)
    X['PeerRelativeIncome'] = X['MonthlyIncome'] / peer
    X['IsLowIncome'] = (X['MonthlyIncome'] < low_income_thresh).astype(int)
    X['IncomePerLevel'] = X['MonthlyIncome'] / X['JobLevel']  # JobLevel >= 1, safe

    # Tenure / career (early tenure + stalled promotion)
    twy = X['TotalWorkingYears'].replace(0, np.nan)  # divide-by-zero guard
    X['EarlyTenure'] = (X['YearsAtCompany'] <= 2).astype(int)
    X['TenureRatio'] = (X['YearsAtCompany'] / twy).fillna(0.0)
    X['PromotionOverdue'] = (X['YearsSinceLastPromotion'] >= 6).astype(int)
    X['JobHoppingIndex'] = (X['NumCompaniesWorked'] / twy).fillna(0.0)

    # Combined high-risk flag (OverTime + low income + early tenure)
    X['IsHighRisk'] = (
        (X['OverTime'] == 1) & (X['IsLowIncome'] == 1) & (X['EarlyTenure'] == 1)
    ).astype(int)

    # Engagement / well-being composites (low scores ~ 2x risk)
    X['SatisfactionComposite'] = X[
        ['JobSatisfaction', 'EnvironmentSatisfaction', 'RelationshipSatisfaction']
    ].mean(axis=1)
    X['EngagementScore'] = X['JobInvolvement'] + X['WorkLifeBalance']
    X['IsLowEngagement'] = (
        (X['JobInvolvement'] <= 2) & (X['WorkLifeBalance'] <= 2)
    ).astype(int)

    # Drop raw helpers now that engineered features are built
    return X.drop(columns=['_JobRole', '_MaritalStatus'])


X_train = engineer(X_train)
X_test = engineer(X_test)
print('After engineering:', X_train.shape, X_test.shape)
assert list(X_train.columns) == list(X_test.columns), 'column mismatch between splits'

## 6. Scaling

`StandardScaler` is fit on **train continuous numerics only**, then applied to
both splits. Binary / flag / dummy columns (values in {0, 1}) are left unscaled.
DataFrame in / DataFrame out keeps column names for downstream SHAP.

In [ ]:
# Columns whose train values are purely {0,1} are flags/dummies -> leave unscaled
binary_cols = [c for c in X_train.columns if X_train[c].dropna().isin([0, 1]).all()]
scale_cols = [c for c in X_train.columns if c not in binary_cols]

scaler = StandardScaler().fit(X_train[scale_cols])
X_train[scale_cols] = scaler.transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

print(f'Scaled {len(scale_cols)} continuous cols | left {len(binary_cols)} flag/dummy cols unscaled')

## 7. Persist outputs

Save modeling-ready matrices (parquet) and the fitted scaler (joblib) so the
modeling notebook can load them directly. Final shape / NaN / parity checks act
as the notebook's built-in verification.

In [ ]:
OUT = Path('processed')
OUT.mkdir(exist_ok=True)

X_train.to_parquet(OUT / 'X_train.parquet')
X_test.to_parquet(OUT / 'X_test.parquet')
y_train.to_frame('Attrition').to_parquet(OUT / 'y_train.parquet')
y_test.to_frame('Attrition').to_parquet(OUT / 'y_test.parquet')
joblib.dump(scaler, OUT / 'scaler.joblib')
joblib.dump(scale_cols, OUT / 'scale_cols.joblib')

# --- Verification ---
assert list(X_train.columns) == list(X_test.columns), 'train/test columns differ'
for name, M in [('X_train', X_train), ('X_test', X_test)]:
    assert not M.isna().any().any(), f'{name} has NaN'
    assert np.isfinite(M.to_numpy(dtype=float)).all(), f'{name} has inf'
assert abs(y_train.mean() - 0.16) < 0.02 and abs(y_test.mean() - 0.16) < 0.02

# Reload check
assert pd.read_parquet(OUT / 'X_train.parquet').shape == X_train.shape

print('Saved to', OUT.resolve())
print(f'X_train {X_train.shape} | X_test {X_test.shape}')
print(f'Attrition rate  train {y_train.mean():.3f}  test {y_test.mean():.3f}')
print('No NaN / inf in final matrices  ✓')

# Sanity: the strongest EDA signal must survive encoding
ot_yes = y_train[X_train['OverTime'] == X_train['OverTime'].max()].mean()
ot_no = y_train[X_train['OverTime'] == X_train['OverTime'].min()].mean()
print(f'Train attrition  OverTime=Yes {ot_yes:.3f}  vs  OverTime=No {ot_no:.3f}  ✓')